### Set Environment Variables

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

### Data Ingestion

In [3]:
os.chdir("../")

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/Agentic_AI.txt")

documents = loader.load()

documents

[Document(metadata={'source': 'data/Agentic_AI.txt'}, page_content='Understanding Agentic AI\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\nKey Characteristics of Agentic AI\nAgentic AI systems are distinct from traditional AI models due to several core characteristics:\n* Goal-Oriented: They possess a clear objective or goal that they strive to achieve.\n* Autonomy: They can operate independently, making decisions and taking actions without continuous human oversight.\n* Perception: They can interpret information from their environment to inform their decisions.\n* Planning and Reasoning: They can formulate plans to reach their goals and reason about the consequ

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

text_chunks = text_splitter.split_documents(documents)

text_chunks

[Document(metadata={'source': 'data/Agentic_AI.txt'}, page_content='Understanding Agentic AI'),
 Document(metadata={'source': 'data/Agentic_AI.txt'}, page_content='Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving'),
 Document(metadata={'source': 'data/Agentic_AI.txt'}, page_content='towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.'),
 Document(metadata={'source': 'data/Agentic_AI.txt'}, page_content='Key Characteristics of Agentic AI\nAgentic AI systems are distinct from traditional AI models due to several core characteristics:'),
 Document(metadata={'source': 'data/Agentic_AI.txt'}, page_content='* Goal-Oriented: They possess a clear objective or goal that they strive to achieve.'),
 Document(me

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

embeddings.embed_query("Hello")

[-0.03313357010483742,
 0.006594267673790455,
 0.007065847050398588,
 -0.08579812198877335,
 -0.019383447244763374,
 0.006432223133742809,
 -0.020736394450068474,
 0.0032028723508119583,
 0.0059592328034341335,
 0.00333585892803967,
 -0.009134207852184772,
 -0.008996911346912384,
 -0.001830110908485949,
 -0.0007971393642947078,
 0.16885443031787872,
 -0.012562962248921394,
 0.006442902144044638,
 0.006800080183893442,
 -0.004335511010140181,
 -0.013132447376847267,
 -0.004231471102684736,
 0.008145235478878021,
 -0.0014682617038488388,
 -7.504242239519954e-05,
 -0.01697615347802639,
 0.003654741682112217,
 0.00782743189483881,
 0.0004786201170645654,
 0.02037719637155533,
 0.007557029370218515,
 0.0011754268780350685,
 -0.005445342510938644,
 -0.006600834894925356,
 0.015904732048511505,
 0.01164266373962164,
 0.005766379646956921,
 0.006837685126811266,
 -0.006046813912689686,
 -0.009320220910012722,
 0.0013290363131090999,
 0.009477527812123299,
 0.0036717706825584173,
 -0.0061006816

In [9]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(text_chunks, embeddings)

vectorstore

In [20]:
retriever = vectorstore.as_retriever()

retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7c5b34bc0850>, search_kwargs={})

In [10]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
Understanding Agentic AI
--------------------------------------------------
Document 2:
Key Characteristics of Agentic AI
Agentic AI systems are distinct from traditional AI models due to several core characteristics:
--------------------------------------------------
Document 3:
Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving
--------------------------------------------------
Document 4:
* Adaptability and Learning: They can adjust their behavior and improve their performance based on new information and experiences.
How Agentic AI Works
--------------------------------------------------


In [11]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [14]:
from langchain_groq import ChatGroq

llm_model = ChatGroq(model="llama-3.1-8b-instant")

In [18]:
from pprint import pprint

test = llm_model.invoke("What is the main obstacles of LLMs?")

pprint(test.content)

('Large Language Models (LLMs) have made significant progress in natural '
 'language processing, but they still face several challenges and obstacles. '
 'Some of the main obstacles of LLMs include:\n'
 '\n'
 '1. **Lack of Common Sense**: LLMs often struggle to understand the nuances '
 'of the real world, such as common sense, humor, and idioms. They may not be '
 'able to reason or make connections between seemingly unrelated pieces of '
 'information.\n'
 '2. **Lack of Contextual Understanding**: While LLMs are great at processing '
 'language, they often struggle to understand the context in which the '
 'language is being used. This can lead to misinterpretation or incorrect '
 'conclusions.\n'
 '3. **Sarcasm and Irony**: LLMs can struggle to detect sarcasm and irony, '
 'which can lead to misinterpretation or incorrect conclusions.\n'
 '4. **Emotional Intelligence**: LLMs lack emotional intelligence, which can '
 'make it difficult for them to understand and respond to emotional

In [19]:
from langchain_classic.schema.output_parser import StrOutputParser

output_parser = StrOutputParser()

In [21]:
from langchain_classic.schema.runnable import RunnablePassthrough

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm_model
    | output_parser
)

In [22]:
rag_chain.invoke("tell me about Agentic AI")

'Agentic AI is a new paradigm in artificial intelligence where systems operate autonomously to achieve goals. Unlike traditional AI models, Agentic AI systems are designed to adapt and learn from new information and experiences. They have the ability to adjust their behavior and improve performance over time. Key characteristics of Agentic AI include adaptability and learning, allowing them to respond effectively to changing situations. This approach enables AI systems to operate more independently and make decisions based on their own judgment. Agentic AI systems are not limited to specific tasks or queries, instead, they can operate in a more dynamic and autonomous manner. Their primary goal is to achieve objectives, rather than just responding to user input. This new paradigm in AI has the potential to revolutionize various industries and applications. The exact mechanisms and techniques used in Agentic AI are still being researched and developed.'